# Feature Extraction: eICU (train) & MIMIC-III (validation)

**Goal:** Extract a harmonized feature set from both datasets so that a model trained on eICU can be externally validated on MIMIC-III.

**Output files:**
- `eicu_features.csv` — training set
- `mimic_features.csv` — validation set

**Feature domains extracted:**
1. Demographics (age, gender, ethnicity)
2. ICU stay metadata (LOS, unit type, admission source)
3. First-24h lab summary stats (min / max / mean for 12 key labs)
4. First-24h vital signs summary stats
5. Diagnosis category (ICD-9 chapter)
6. Target label: ICU mortality (binary)

# 0. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
EICU_DIR  = '/content/drive/MyDrive/AI in Medicine/data/eicu_raw/'
MIMIC_DIR = '/content/drive/MyDrive/AI in Medicine/data/mimic_3_raw/mimic/'
EICU_OUT_DIR   = '/content/drive/MyDrive/AI in Medicine/data/output_data/eicu_train/'
MIMIC_OUT_DIR   = '/content/drive/MyDrive/AI in Medicine/data/output_data/mimic_val/'


import os
os.makedirs(OUT_DIR, exist_ok=True)
print('Paths configured.')

Paths configured.


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)
print('Libraries loaded.')

Libraries loaded.


# 1.1. eICU Feature Extraction


In [ ]:
def load_eicu(fname, **kwargs):
    """Load eICU CSV."""
    path = os.path.join(EICU_DIR, fname)

    for ext in ['.csv']:
        full = path + ext if not path.endswith(ext) else path
        if os.path.exists(full):
            return pd.read_csv(full, **kwargs)
    raise FileNotFoundError(f'Cannot find {fname} in {EICU_DIR}')

# ── Patient (core demographics + stay info)
eicu_pt = load_eicu('patient', low_memory=False)
eicu_pt.columns = eicu_pt.columns.str.lower()
print(f'patient rows: {len(eicu_pt):,}  |  cols: {list(eicu_pt.columns)}')

patient rows: 2,520  |  cols: ['patientunitstayid', 'patienthealthsystemstayid', 'gender', 'age', 'ethnicity', 'hospitalid', 'wardid', 'apacheadmissiondx', 'admissionheight', 'hospitaladmittime24', 'hospitaladmitoffset', 'hospitaladmitsource', 'hospitaldischargeyear', 'hospitaldischargetime24', 'hospitaldischargeoffset', 'hospitaldischargelocation', 'hospitaldischargestatus', 'unittype', 'unitadmittime24', 'unitadmitsource', 'unitvisitnumber', 'unitstaytype', 'admissionweight', 'dischargeweight', 'unitdischargetime24', 'unitdischargeoffset', 'unitdischargelocation', 'unitdischargestatus', 'uniquepid']


In [ ]:
# ── Lab results
eicu_lab = load_eicu('lab', low_memory=False)
eicu_lab.columns = eicu_lab.columns.str.lower()
print(f'lab rows: {len(eicu_lab):,}')
eicu_lab.head(3)

lab rows: 434,660


,labid,patientunitstayid,labresultoffset,labtypeid,labname,labresult,labresulttext,labmeasurenamesystem,labmeasurenameinterface,labresultrevisedoffset
0,437880563,1754323,-647,3,Hct,38.300,38.3,%,%,-631
1,437880572,1754323,-647,3,platelets x 1000,181.000,181,K/mcL,k/mm cu,-631
2,437880560,1754323,-647,3,RBC,4.860,4.86,M/mcL,m/mm cu,-631


In [ ]:
# ── Vital signs (periodic = 5-min medians)
eicu_vp = load_eicu('vitalPeriodic', low_memory=False)
eicu_vp.columns = eicu_vp.columns.str.lower()
print(f'vitalPeriodic rows: {len(eicu_vp):,}')
print(eicu_vp.columns.tolist())

vitalPeriodic rows: 1,634,960
['vitalperiodicid', 'patientunitstayid', 'observationoffset', 'temperature', 'sao2', 'heartrate', 'respiration', 'cvp', 'etco2', 'systemicsystolic', 'systemicdiastolic', 'systemicmean', 'pasystolic', 'padiastolic', 'pamean', 'st1', 'st2', 'st3', 'icp']


In [ ]:
# ── Diagnosis
eicu_dx = load_eicu('diagnosis', low_memory=False)
eicu_dx.columns = eicu_dx.columns.str.lower()
print(f'diagnosis rows: {len(eicu_dx):,}')
eicu_dx.head(3)

diagnosis rows: 24,978


,diagnosisid,patientunitstayid,activeupondischarge,diagnosisoffset,diagnosisstring,icd9code,diagnosispriority
0,7607199,346380,False,5028,cardiovascular|ventricular disorders|hypertension,"401.9, I10",Other
1,7570429,346380,False,685,neurologic|altered mental status / pain|change...,"780.09, R41.82",Major
2,7705483,346380,True,5035,cardiovascular|shock / hypotension|hypotension,"458.9, I95.9",Major


## 1.2 Demographics & stay metadata

In [ ]:
# ── Age
# eICU stores age as a string; '>89' patients are de-identified → recode to 90
eicu_pt['age_clean'] = (
    eicu_pt['age']
    .astype(str)
    .str.replace('> 89', '90', regex=False)
    .str.replace('>89',  '90', regex=False)
    .str.strip()
)
eicu_pt['age_clean'] = pd.to_numeric(eicu_pt['age_clean'], errors='coerce')

# ── Gender
eicu_pt['gender_binary'] = eicu_pt['gender'].map({'Male': 1, 'Female': 0})

# ── Ethnicity → 5 harmonized groups
ETH_MAP_EICU = {
    'Caucasian':              'White',
    'African American':       'African',
    'Hispanic':               'Hispanic',
    'Asian':                  'Asian',
    'Native American':        'American',
    'Other/Unknown':          'Unknown',
    '':                       'Unknown',
}
eicu_pt['ethnicity_grp'] = (
    eicu_pt['ethnicity']
    .fillna('Other/Unknown')
    .map(lambda x: ETH_MAP_EICU.get(x, 'Other'))
)

# ── ICU LOS (days)
# unitdischargeoffset is in minutes from ICU admission
eicu_pt['icu_los_days'] = eicu_pt['unitdischargeoffset'] / 1440
eicu_pt.loc[eicu_pt['icu_los_days'] < 0, 'icu_los_days'] = np.nan  # sanity

# ── Unit type → harmonized care-unit category
UNIT_MAP_EICU = {
    'MICU':          'MICU',
    'CCU-CTICU':     'CCU',
    'Cardiac ICU':   'CCU',
    'CSICU':         'CSICU',
    'SICU':          'SICU',
    'NEURO SICU':    'NSICU',
    'Med-Surg ICU':  'MICU',
}
eicu_pt['careunit'] = eicu_pt['unittype'].map(lambda x: UNIT_MAP_EICU.get(str(x), 'Other'))

# ── Admission source
ADMIT_SRC_EICU = {
    'Emergency Department': 'Emergency',
    'Direct Admit':         'Direct',
    'Floor':                'Floor',
    'Operating Room':       'Operating Room',
    'Recovery Room':        'Recovery Room',
    'Other ICU':            'Transfer',
    'Other Hospital':       'Transfer',
    'Chest Pain Center':    'Chest Pain',
}
eicu_pt['admit_source'] = eicu_pt['hospitaladmitsource'].map(
    lambda x: ADMIT_SRC_EICU.get(str(x), 'Unknown')
)

# ── Mortality label
eicu_pt['mortality'] = (eicu_pt['unitdischargestatus'] == 'Expired').astype(int)

print('eICU demographics processed.')
print(eicu_pt[['age_clean','gender_binary','ethnicity_grp','icu_los_days',
               'careunit','admit_source','mortality']].describe())

eICU demographics processed.
       age_clean  gender_binary  icu_los_days  mortality
count   2516.000       2516.000      2520.000   2520.000
mean      63.281          0.599         2.419      0.050
std       17.765          0.490         3.456      0.218
min       15.000          0.000         0.000      0.000
25%       53.000          0.000         0.790      0.000
50%       66.000          1.000         1.472      0.000
75%       77.000          1.000         2.777      0.000
max       90.000          1.000        46.180      1.000


## 1.3 First-24h lab features

In [ ]:
# labresultoffset: minutes from ICU admission (negative = before admission)
# Keep only first 24 hours (0–1440 min) to match MIMIC's first-24h window

LAB_TARGETS = {
    # eICU label            : harmonized column prefix
    'creatinine':            'creatinine',
    'BUN':                   'bun',
    'sodium':                'sodium',
    'potassium':             'potassium',
    'glucose':               'glucose',
    'Hgb':                   'hemoglobin',
    'Hct':                   'hematocrit',
    'WBC x 1000':            'wbc',
    'platelets x 1000':      'platelets',
    'bicarbonate':           'bicarbonate',
    'ALT (SGPT)':            'alt',
    'albumin':               'albumin',
    'lactate':               'lactate',
    'PT - INR':              'inr',
}

lab24 = eicu_lab[
    (eicu_lab['labresultoffset'] >= 0) &
    (eicu_lab['labresultoffset'] <= 1440) &
    (eicu_lab['labname'].isin(LAB_TARGETS.keys()))
].copy()

lab24['lab_col'] = lab24['labname'].map(LAB_TARGETS)
lab24['labresult'] = pd.to_numeric(lab24['labresult'], errors='coerce')

# Aggregate: min, max, mean per stay per lab
lab_agg = (
    lab24
    .groupby(['patientunitstayid', 'lab_col'])['labresult']
    .agg(['min', 'max', 'mean'])
    .reset_index()
)
lab_agg.columns = ['patientunitstayid', 'lab_col', 'lab_min', 'lab_max', 'lab_mean']

# Pivot to wide format: one row per stay
lab_wide = lab_agg.pivot_table(
    index='patientunitstayid',
    columns='lab_col',
    values=['lab_min', 'lab_max', 'lab_mean']
)
# Flatten column multi-index: e.g. ('lab_min', 'creatinine') → 'creatinine_min'
lab_wide.columns = [f'{col}_{stat.replace("lab_","")}' for stat, col in lab_wide.columns]
lab_wide = lab_wide.reset_index()

print(f'Lab feature shape: {lab_wide.shape}')
print(lab_wide.columns.tolist()[:10], '...')

Lab feature shape: (2208, 43)
['patientunitstayid', 'albumin_max', 'alt_max', 'bicarbonate_max', 'bun_max', 'creatinine_max', 'glucose_max', 'hematocrit_max', 'hemoglobin_max', 'inr_max'] ...


## 1.4 First-24h vital sign features

In [ ]:
# vitalperiodicoffset = minutes from ICU admission
VITAL_COLS = {
    'heartrate':         'heartrate',
    'respiration':       'respiration',
    'sao2':              'sao2',
    'temperature':       'temp',
    'systemicsystolic':  'ssystolic',
    'systemicdiastolic': 'sdiastolic',
    'systemicmean':      'systemicmean',
}

vp24 = eicu_vp[
    (eicu_vp['observationoffset'] >= 0) &
    (eicu_vp['observationoffset'] <= 1440)
].copy()

vital_agg_frames = []
for raw_col, prefix in VITAL_COLS.items():
    if raw_col not in vp24.columns:
        print(f'  WARNING: {raw_col} not found in vitalPeriodic — skipping')
        continue
    tmp = (
        vp24[['patientunitstayid', raw_col]]
        .replace(-1, np.nan)          # eICU uses -1 as missing sentinel
        .dropna(subset=[raw_col])
        .groupby('patientunitstayid')[raw_col]
        .agg(['min','max','mean'])
        .rename(columns={'min': f'{prefix}_min', 'max': f'{prefix}_max', 'mean': f'{prefix}_mean'})
    )
    vital_agg_frames.append(tmp)

vital_wide = pd.concat(vital_agg_frames, axis=1).reset_index()
print(f'Vital feature shape: {vital_wide.shape}')
print(vital_wide.columns.tolist())

Vital feature shape: (2370, 22)
['patientunitstayid', 'heartrate_min', 'heartrate_max', 'heartrate_mean', 'respiration_min', 'respiration_max', 'respiration_mean', 'sao2_min', 'sao2_max', 'sao2_mean', 'temp_min', 'temp_max', 'temp_mean', 'ssystolic_min', 'ssystolic_max', 'ssystolic_mean', 'sdiastolic_min', 'sdiastolic_max', 'sdiastolic_mean', 'systemicmean_min', 'systemicmean_max', 'systemicmean_mean']


## 1.5 Diagnosis ICD-9





In [ ]:
def icd9_to_chapter(code):
    """Map ICD-9 code string to chapter name (broad disease category)."""
    try:
        # Remove decimal and leading zeros; handle V/E codes
        c = str(code).strip().upper().replace('.', '')
        if c.startswith('V'):
            return 'Supplementary'
        if c.startswith('E'):
            return 'External causes'
        n = int(c[:3])
    except (ValueError, TypeError):
        return 'Unknown'

    chapters = [
        (1,   139,  'Infectious & parasitic'),
        (140, 239,  'Neoplasms'),
        (240, 279,  'Endocrine & metabolic'),
        (280, 289,  'Blood disorders'),
        (290, 319,  'Mental disorders'),
        (320, 389,  'Nervous system'),
        (390, 459,  'Circulatory'),
        (460, 519,  'Respiratory'),
        (520, 579,  'Digestive'),
        (580, 629,  'Genitourinary'),
        (630, 679,  'Pregnancy/childbirth'),
        (680, 709,  'Skin'),
        (710, 739,  'Musculoskeletal'),
        (740, 759,  'Congenital'),
        (760, 779,  'Perinatal'),
        (780, 799,  'Symptoms & signs'),
        (800, 999,  'Injury & poisoning'),
    ]
    for lo, hi, name in chapters:
        if lo <= n <= hi:
            return name
    return 'Unknown'

# eICU diagnosis table has 'icd9code' column
# Take the primary diagnosis (diagnosispriority == 'Primary' or lowest sequence number)
eicu_dx['icd9code'] = eicu_dx['icd9code'].astype(str).str.strip()

# Keep primary diagnosis per stay (diagnosispriority = 'Primary' if available)
primary_dx = (
    eicu_dx[eicu_dx['diagnosispriority'].str.lower().str.contains('primary', na=False)]
    if 'diagnosispriority' in eicu_dx.columns
    else eicu_dx.sort_values('diagnosisid').groupby('patientunitstayid').first().reset_index()
)

dx_first = (
    primary_dx
    .sort_values('diagnosisid' if 'diagnosisid' in primary_dx.columns else primary_dx.columns[0])
    .groupby('patientunitstayid')['icd9code']
    .first()
    .reset_index()
)
dx_first['icd9_chapter'] = dx_first['icd9code'].apply(icd9_to_chapter)
dx_first = dx_first[['patientunitstayid', 'icd9_chapter']]

print(dx_first['icd9_chapter'].value_counts())

icd9_chapter
Circulatory               400
Unknown                   327
Respiratory               250
Symptoms & signs          147
Injury & poisoning        135
Digestive                 122
Endocrine & metabolic     117
Infectious & parasitic     93
Genitourinary              32
Mental disorders           23
Nervous system             22
Neoplasms                  17
External causes             9
Blood disorders             6
Skin                        5
Supplementary               2
Pregnancy/childbirth        2
Musculoskeletal             2
Name: count, dtype: int64


## 1.6 Merge all eICU features

In [ ]:
# Base: demographics + stay metadata + label
DEMO_COLS = [
    'patientunitstayid',
    'age_clean', 'gender_binary', 'ethnicity_grp',
    'icu_los_days', 'careunit', 'admit_source',
    'mortality'
]
eicu_base = eicu_pt[DEMO_COLS].copy()

eicu_features = (
    eicu_base
    .merge(lab_wide,   on='patientunitstayid', how='left')
    .merge(vital_wide, on='patientunitstayid', how='left')
    .merge(dx_first,   on='patientunitstayid', how='left')
)

# Rename id column for clarity
eicu_features.rename(columns={'patientunitstayid': 'stay_id'}, inplace=True)
eicu_features['dataset'] = 'eicu'

print(f'eICU feature table: {eicu_features.shape}')
print(f'Mortality rate: {eicu_features["mortality"].mean():.3f}')
eicu_features.head(3)

eICU feature table: (2520, 73)
Mortality rate: 0.050


,stay_id,age_clean,gender_binary,ethnicity_grp,icu_los_days,careunit,admit_source,mortality,albumin_max,alt_max,bicarbonate_max,bun_max,creatinine_max,glucose_max,hematocrit_max,hemoglobin_max,inr_max,lactate_max,platelets_max,potassium_max,sodium_max,wbc_max,albumin_mean,alt_mean,bicarbonate_mean,...,sodium_min,wbc_min,heartrate_min,heartrate_max,heartrate_mean,respiration_min,respiration_max,respiration_mean,sao2_min,sao2_max,sao2_mean,temp_min,temp_max,temp_mean,ssystolic_min,ssystolic_max,ssystolic_mean,sdiastolic_min,sdiastolic_max,sdiastolic_mean,systemicmean_min,systemicmean_max,systemicmean_mean,icd9_chapter,dataset
0,141764,87.000,0.000,White,0.239,MICU,Unknown,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,90.000,138.000,106.652,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,eicu
1,141765,87.000,0.000,White,1.562,MICU,Unknown,0,NaN,NaN,21.000,28.000,1.040,61.000,37.800,12.300,NaN,NaN,191.000,4.100,139.000,10.200,NaN,NaN,21.000,...,139.000,10.200,72.000,116.000,83.191,17.000,39.000,24.783,95.000,98.000,96.609,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,eicu
2,143870,76.000,1.000,White,0.551,SICU,Operating Room,0,NaN,NaN,25.000,14.000,1.140,140.000,34.100,12.300,NaN,NaN,152.000,4.400,139.000,11.700,NaN,NaN,23.500,...,133.000,11.700,40.000,55.000,45.449,45.000,86.000,67.032,80.000,100.000,96.285,NaN,NaN,NaN,53.000,138.000,109.816,36.000,57.000,42.715,47.000,79.000,60.835,Unknown,eicu


# 2. MIMIC-III Feature Extraction

## 2.1 Load core MIMIC-III tables

**Bugs fixed vs original version:**
1. **Age overflow** — `dob` subtraction replaced with year-arithmetic `.apply()`
2. **ALT label** — was `'Alanine Aminotransferase (ALT)'` (no match); correct label in D_LABITEMS is `'Alanine Aminotransferase (ALT)'` with itemid 50861 — fixed with exact itemid filter
3. **INR label** — was `'INR(PT)'` (no match via substring on wrong string); correct itemid is 51237 — fixed
4. **Lab join** — `LABEVENTS` has NO `icustay_id`; join must go via `hadm_id` → `ICUSTAYS.intime` for 24h window filter
5. **Blood-only filtering** — multiple labs (e.g. Sodium, Creatinine) exist for multiple fluids; we now filter to `fluid = 'Blood'` to avoid duplicates
6. **Hemoglobin false matches** — substring `'Hemoglobin'` matched `Carboxyhemoglobin`, `Methemoglobin`; now uses exact label
7. **Sodium/Potassium** — matched wrong fluid variants; now blood-only via itemid whitelist

In [ ]:
def load_mimic(fname):
    for ext in ['.csv', '.csv.gz']:
        path = os.path.join(MIMIC_DIR, fname + ext)
        if os.path.exists(path):
            df = pd.read_csv(path, low_memory=False)
            df.columns = df.columns.str.lower()
            print(f'  {fname}: {len(df):,} rows')
            return df
    raise FileNotFoundError(f'{fname} not found in {MIMIC_DIR}')

print('Loading tables...')
patients   = load_mimic('PATIENTS')
admissions = load_mimic('ADMISSIONS')
icustays   = load_mimic('ICUSTAYS')
labevents  = load_mimic('LABEVENTS')
d_labitems = load_mimic('D_LABITEMS')
diagnoses  = load_mimic('DIAGNOSES_ICD')

Loading tables...
  PATIENTS: 100 rows
  ADMISSIONS: 129 rows
  ICUSTAYS: 136 rows
  LABEVENTS: 76,074 rows
  D_LABITEMS: 753 rows
  DIAGNOSES_ICD: 1,761 rows


## 2. Build one base row per ICU stay

The 3-level key: `subject_id` → `hadm_id` → `icustay_id`  
We start from `ICUSTAYS` (one row per ICU stay) and join outward.

In [ ]:
# Parse datetimes once
icustays['intime']  = pd.to_datetime(icustays['intime'])
icustays['outtime'] = pd.to_datetime(icustays['outtime'])
admissions['admittime'] = pd.to_datetime(admissions['admittime'])
patients['dob'] = pd.to_datetime(patients['dob'])

# Join ICUSTAYS ← ADMISSIONS ← PATIENTS
base = (
    icustays
    .merge(
        admissions[['hadm_id','subject_id','admittime',
                    'ethnicity','admission_type','admission_location',
                    'hospital_expire_flag']],
        on=['hadm_id','subject_id'], how='left'
    )
    .merge(
        patients[['subject_id','gender','dob']],
        on='subject_id', how='left'
    )
)
print(f'Base table: {base.shape}  (one row per ICU stay)')
print(base[['icustay_id','hadm_id','subject_id','intime','dob']].head(3))

Base table: (136, 19)  (one row per ICU stay)
   icustay_id  hadm_id  subject_id              intime        dob
0      206504   142345       10006 2164-10-23 21:10:15 2094-03-05
1      232110   105331       10011 2126-08-14 22:34:00 2090-06-05
2      264446   165520       10013 2125-10-04 23:38:00 2038-09-03


## 2.2 Demographics & stay metadata

In [ ]:
def calc_age(intime, dob):
    """Year-arithmetic age — no timedelta, no overflow."""
    if pd.isna(intime) or pd.isna(dob):
        return np.nan
    age = intime.year - dob.year
    # Subtract 1 if birthday hasn't happened yet in the admission year
    if (intime.month, intime.day) < (dob.month, dob.day):
        age -= 1
    return age

base['age_clean'] = base.apply(
    lambda r: calc_age(r['intime'], r['dob']), axis=1
)

# Recode de-identified age (~300) to 90
base.loc[base['age_clean'] > 120, 'age_clean'] = 90
base.loc[base['age_clean'] < 0,   'age_clean'] = np.nan

print('Age distribution:')
print(base['age_clean'].describe())
print(f"  Patients recoded to 90: {(base['age_clean'] == 90).sum()}")

Age distribution:
count   136.000
mean     70.382
std      16.190
min      17.000
25%      63.000
50%      74.000
75%      83.000
max      90.000
Name: age_clean, dtype: float64
  Patients recoded to 90: 9


In [ ]:
# ── Gender ────────────────────────────────────────────────────────────────────
base['gender_binary'] = base['gender'].map({'M': 1, 'F': 0})

# ── Ethnicity → 5 harmonized groups (matching eICU categories) ────────────────
def map_ethnicity(eth):
    eth = str(eth).upper()
    if 'WHITE' in eth:                        return 'White'
    if 'BLACK' in eth or 'AFRICAN' in eth:    return 'African'
    if 'HISPANIC' in eth or 'LATINO' in eth:  return 'Hispanic'
    if 'ASIAN' in eth:                        return 'Asian'
    if 'UNKNOWN' in eth or 'UNABLE' in eth:   return 'Unknown'
    return 'Other'

base['ethnicity_grp'] = base['ethnicity'].apply(map_ethnicity)

# ── ICU LOS in days — already stored in ICUSTAYS.los
# los is in DAYS (confirmed from data: 1.6325, 13.8507 etc.)
base['icu_los_days'] = base['los'].astype(float)
base.loc[base['icu_los_days'] < 0, 'icu_los_days'] = np.nan

# ── Care unit — harmonize to match eICU labels
# MIMIC values: MICU, CCU, CSRU, SICU, TSICU, NICU, NWARD
UNIT_MAP = {'MICU':'MICU','CCU':'CCU','CSRU':'CSICU',
            'SICU':'SICU','TSICU':'SICU','NICU':'NSICU','NWARD':'NSICU'}
base['careunit'] = base['first_careunit'].map(lambda x: UNIT_MAP.get(str(x), 'Other'))

# ── Admission source — harmonize to match eICU labels ─────────────────────────
# MIMIC values from ADMISSIONS.admission_location
def map_admit_src(src):
    src = str(src).upper()
    if 'EMERGENCY' in src:                          return 'Emergency'
    if 'ELECTIVE' in src or 'CLINIC' in src:        return 'Elective'
    if 'OPERATING' in src or 'OR ' in src:          return 'Operating Room'
    if 'TRANSFER' in src or 'OTHER HOSP' in src:    return 'Transfer'
    if 'FLOOR' in src or 'WARD' in src:             return 'Floor'
    return 'Unknown'

base['admit_source'] = base['admission_location'].apply(map_admit_src)

# ── Mortality label
# hospital_expire_flag = 1 means the patient died during this hospital admission
base['mortality'] = base['hospital_expire_flag'].astype(int)

print('Demographics done.')
print(base[['age_clean','gender_binary','ethnicity_grp',
             'icu_los_days','careunit','admit_source','mortality']].head(5))

Demographics done.
   age_clean  gender_binary ethnicity_grp  icu_los_days careunit admit_source  \
0     70.000              0       African         1.633     MICU    Emergency   
1     36.000              0       Unknown        13.851     MICU     Transfer   
2     87.000              0       Unknown         2.650     MICU     Transfer   
3     73.000              0         White         2.144      CCU    Emergency   
4     48.000              1         White         1.294     MICU     Transfer   

   mortality  
0          0  
1          1  
2          1  
3          0  
4          1  


## 2.3 First-24h lab features

In [ ]:
# ── Build exact itemid whitelist from D_LABITEMS ──────────────────────────────
# Using itemids instead of label substrings avoids false matches
# (e.g. 'Hemoglobin' would also match 'Carboxyhemoglobin')

# Verified against actual D_LABITEMS (Blood fluid only)
LAB_ITEMIDS = {
    50912: 'creatinine',      # Creatinine, Blood
    51006: 'bun',             # Urea Nitrogen, Blood
    50983: 'sodium',          # Sodium, Blood
    50971: 'potassium',       # Potassium, Blood
    50931: 'glucose',         # Glucose, Blood
    50811: 'hemoglobin',      # Hemoglobin, Blood (not Carboxy/Meth)
    51221: 'hematocrit',      # Hematocrit, Blood
    51301: 'wbc',             # White Blood Cells, Blood
    51265: 'platelets',       # Platelet Count, Blood
    50882: 'bicarbonate',     # Bicarbonate, Blood
    50861: 'alt',             # Alanine Aminotransferase (ALT), Blood
    50862: 'albumin',         # Albumin, Blood
    50813: 'lactate',         # Lactate, Blood
    51237: 'inr',             # INR(PT), Blood
}

# Verify all itemids exist in D_LABITEMS
found = d_labitems[d_labitems['itemid'].isin(LAB_ITEMIDS.keys())][['itemid','label','fluid']]
print(f'Verified {len(found)}/14 lab itemids in D_LABITEMS:')
print(found.to_string(index=False))

Verified 14/14 lab itemids in D_LABITEMS:
 itemid                          label fluid
  50811                     Hemoglobin Blood
  50813                        Lactate Blood
  50861 Alanine Aminotransferase (ALT) Blood
  50862                        Albumin Blood
  50882                    Bicarbonate Blood
  50912                     Creatinine Blood
  50931                        Glucose Blood
  50971                      Potassium Blood
  50983                         Sodium Blood
  51006                  Urea Nitrogen Blood
  51221                     Hematocrit Blood
  51237                        INR(PT) Blood
  51265                 Platelet Count Blood
  51301              White Blood Cells Blood


In [ ]:
# ── Filter LABEVENTS ─────────────────────────────────────────────────────────
labevents['charttime'] = pd.to_datetime(labevents['charttime'])

# Drop rows with no hadm_id (outpatient labs — not linked to a hospital stay)
lab = labevents.dropna(subset=['hadm_id']).copy()
lab['hadm_id'] = lab['hadm_id'].astype(int)

# Keep only our 14 target labs
lab = lab[lab['itemid'].isin(LAB_ITEMIDS.keys())].copy()
lab['lab_col'] = lab['itemid'].map(LAB_ITEMIDS)
lab['valuenum'] = pd.to_numeric(lab['valuenum'], errors='coerce')

print(f'Lab rows after filtering: {len(lab):,}')
print(lab['lab_col'].value_counts())

Lab rows after filtering: 17,899
lab_col
potassium      1820
hematocrit     1800
sodium         1746
bicarbonate    1716
creatinine     1710
bun            1704
glucose        1703
platelets      1598
wbc            1544
inr            1104
lactate         542
alt             494
albumin         299
hemoglobin      119
Name: count, dtype: int64


In [ ]:
# ── Join to ICUSTAYS to get the 24h window ─────────────────────────────────
# LABEVENTS has hadm_id but NOT icustay_id.
# A single hadm_id can have >1 ICU stay — we match lab to the correct stay
# by checking whether charttime falls within [intime, intime+24h].

icu_times = icustays[['icustay_id','hadm_id','intime']].copy()

lab24 = lab.merge(icu_times, on='hadm_id', how='inner')

# Compute hours since ICU admission
lab24['hours_since_icu'] = (
    (lab24['charttime'] - lab24['intime']).dt.total_seconds() / 3600
)

# Keep only first 24h (0 to 24h from ICU admission)
lab24 = lab24[(lab24['hours_since_icu'] >= 0) & (lab24['hours_since_icu'] <= 24)]

print(f'Lab rows in first 24h: {len(lab24):,}')
print(f'ICU stays with ≥1 lab: {lab24["icustay_id"].nunique()} / {icustays["icustay_id"].nunique()}')

Lab rows in first 24h: 2,727
ICU stays with ≥1 lab: 135 / 136


In [ ]:
# ── Aggregate: min, max, mean per ICU stay per lab
lab_agg = (
    lab24
    .groupby(['icustay_id', 'lab_col'])['valuenum']
    .agg(['min', 'max', 'mean'])
    .reset_index()
)
lab_agg.columns = ['icustay_id', 'lab_col', 'lab_min', 'lab_max', 'lab_mean']

# Pivot long → wide: one row per ICU stay, columns = creatinine_min, creatinine_max ...
lab_wide = lab_agg.pivot_table(
    index='icustay_id',
    columns='lab_col',
    values=['lab_min', 'lab_max', 'lab_mean']
)
# Flatten multi-index: ('lab_min', 'creatinine') → 'creatinine_min'
lab_wide.columns = [f'{col}_{stat.replace("lab_","")}' for stat, col in lab_wide.columns]
lab_wide = lab_wide.reset_index()

print(f'Lab feature shape: {lab_wide.shape}')  # should be ~136 stays × 43 cols
print('Columns:', sorted(lab_wide.columns.tolist()))

Lab feature shape: (135, 43)
Columns: ['albumin_max', 'albumin_mean', 'albumin_min', 'alt_max', 'alt_mean', 'alt_min', 'bicarbonate_max', 'bicarbonate_mean', 'bicarbonate_min', 'bun_max', 'bun_mean', 'bun_min', 'creatinine_max', 'creatinine_mean', 'creatinine_min', 'glucose_max', 'glucose_mean', 'glucose_min', 'hematocrit_max', 'hematocrit_mean', 'hematocrit_min', 'hemoglobin_max', 'hemoglobin_mean', 'hemoglobin_min', 'icustay_id', 'inr_max', 'inr_mean', 'inr_min', 'lactate_max', 'lactate_mean', 'lactate_min', 'platelets_max', 'platelets_mean', 'platelets_min', 'potassium_max', 'potassium_mean', 'potassium_min', 'sodium_max', 'sodium_mean', 'sodium_min', 'wbc_max', 'wbc_mean', 'wbc_min']


## 2.4 First-24h vital signs (CHARTEVENTS)

> **Note:** `CHARTEVENTS.csv` is very large (~30 GB for full MIMIC). The demo subset included in your files may not have it. If the file is absent, this cell will be skipped gracefully and vital columns will be NaN — you can still proceed with labs + demographics.

In [ ]:
# CHARTEVENTS is ~33GB in full MIMIC; the demo may not include it.
# Loaded in chunks, filtering to only vital itemids to stay memory-safe.

# Both CareVue (CV) and MetaVision (MV) itemids for the same vital
VITAL_ITEMIDS = {
    211:    'hr',   220045: 'hr',
    646:    'spo2', 220277: 'spo2',
    618:    'rr',   220210: 'rr',   3603: 'rr',
    223762: 'temp', 676:    'temp',
    51:     'sbp',  442: 'sbp', 455: 'sbp', 6701: 'sbp', 220179: 'sbp', 220050: 'sbp',
    8368:   'dbp',  8440: 'dbp', 8441: 'dbp', 8555: 'dbp', 220180: 'dbp', 220051: 'dbp',
    52:     'map',  456: 'map', 6702: 'map', 220052: 'map', 220181: 'map', 225312: 'map',
}

chartevents_path = None
for ext in ['.csv', '.csv.gz']:
    p = os.path.join(MIMIC_DIR, 'CHARTEVENTS' + ext)
    if os.path.exists(p):
        chartevents_path = p
        break

vital_wide = icustays[['icustay_id']].copy()  # default: empty, all NaN

if chartevents_path:
    print(f'Loading CHARTEVENTS from {chartevents_path} (chunked)...')
    chunks = []
    for chunk in pd.read_csv(
        chartevents_path,
        usecols=['icustay_id','charttime','itemid','valuenum','error'],
        chunksize=500_000,
        low_memory=False
    ):
        chunk.columns = chunk.columns.str.lower()
        chunk = chunk[
            chunk['itemid'].isin(VITAL_ITEMIDS.keys()) &
            (chunk['error'].fillna(0) == 0) &
            chunk['icustay_id'].notna()
        ]
        chunks.append(chunk)
    ce = pd.concat(chunks, ignore_index=True)
    print(f'  Vital rows loaded: {len(ce):,}')

    ce['vital_col']  = ce['itemid'].map(VITAL_ITEMIDS)
    ce['charttime']  = pd.to_datetime(ce['charttime'])
    ce['icustay_id'] = ce['icustay_id'].astype(int)

    # Join ICU intime for 24h window
    ce = ce.merge(icustays[['icustay_id','intime']], on='icustay_id', how='inner')
    ce['hours_since_icu'] = (ce['charttime'] - ce['intime']).dt.total_seconds() / 3600
    ce24 = ce[(ce['hours_since_icu'] >= 0) & (ce['hours_since_icu'] <= 24)]

    vital_agg = (
        ce24.groupby(['icustay_id','vital_col'])['valuenum']
        .agg(['min','max','mean'])
        .reset_index()
    )
    vital_agg.columns = ['icustay_id','vital_col','vit_min','vit_max','vit_mean']

    vital_wide = vital_agg.pivot_table(
        index='icustay_id',
        columns='vital_col',
        values=['vit_min','vit_max','vit_mean']
    )
    vital_wide.columns = [f'{col}_{stat.replace("vit_","")}' for stat, col in vital_wide.columns]
    vital_wide = vital_wide.reset_index()
    print(f'Vital feature shape: {vital_wide.shape}')
else:
    print('CHARTEVENTS not found — vital columns will be NaN (demo mode).')

Loading CHARTEVENTS from /content/drive/MyDrive/AI in Medicine/data/mimic_3_raw/mimic/CHARTEVENTS.csv (chunked)...
  Vital rows loaded: 94,137
Vital feature shape: (132, 22)


## 2.5 Diagnosis ICD-9 chapter

In [ ]:
def icd9_to_chapter(code):
    """Maps ICD-9 code to broad clinical chapter (same function used in eICU)."""
    try:
        c = str(code).strip().upper().replace('.', '')
        if c.startswith('V'): return 'Supplementary'
        if c.startswith('E'): return 'External causes'
        n = int(c[:3])
    except (ValueError, TypeError):
        return 'Unknown'
    chapters = [
        (1,  139,  'Infectious & parasitic'),
        (140,239,  'Neoplasms'),
        (240,279,  'Endocrine & metabolic'),
        (280,289,  'Blood disorders'),
        (290,319,  'Mental disorders'),
        (320,389,  'Nervous system'),
        (390,459,  'Circulatory'),
        (460,519,  'Respiratory'),
        (520,579,  'Digestive'),
        (580,629,  'Genitourinary'),
        (630,679,  'Pregnancy/childbirth'),
        (680,709,  'Skin'),
        (710,739,  'Musculoskeletal'),
        (740,759,  'Congenital'),
        (760,779,  'Perinatal'),
        (780,799,  'Symptoms & signs'),
        (800,999,  'Injury & poisoning'),
    ]
    for lo, hi, name in chapters:
        if lo <= n <= hi: return name
    return 'Unknown'

# seq_num == 1 is the primary diagnosis in MIMIC-III
primary_dx = diagnoses[diagnoses['seq_num'] == 1][['hadm_id','icd9_code']].copy()
primary_dx['icd9_chapter'] = primary_dx['icd9_code'].apply(icd9_to_chapter)

# Link diagnosis to ICU stay via hadm_id
icu_dx = icustays[['icustay_id','hadm_id']].merge(
    primary_dx[['hadm_id','icd9_chapter']], on='hadm_id', how='left'
)[['icustay_id','icd9_chapter']]

print(icu_dx['icd9_chapter'].value_counts())

icd9_chapter
Circulatory               28
Infectious & parasitic    26
Digestive                 20
Respiratory               19
Injury & poisoning        17
Neoplasms                 16
Genitourinary              5
Endocrine & metabolic      3
Nervous system             1
Musculoskeletal            1
Name: count, dtype: int64


## 2.6 Merge all MIMIC-III features

In [ ]:
DEMO_COLS = [
    'icustay_id', 'hadm_id',
    'age_clean', 'gender_binary', 'ethnicity_grp',
    'icu_los_days', 'careunit', 'admit_source',
    'mortality'
]

mimic_features = (
    base[DEMO_COLS]
    .merge(lab_wide,   on='icustay_id', how='left')
    .merge(vital_wide, on='icustay_id', how='left')
    .merge(icu_dx,     on='icustay_id', how='left')
)

mimic_features.rename(columns={'icustay_id': 'stay_id'}, inplace=True)
mimic_features['dataset'] = 'mimic'

print(f'\nFinal shape: {mimic_features.shape}')
print(f'Mortality rate: {mimic_features["mortality"].mean():.3f}')
mimic_features.head(3)


Final shape: (136, 74)
Mortality rate: 0.338


,stay_id,hadm_id,age_clean,gender_binary,ethnicity_grp,icu_los_days,careunit,admit_source,mortality,albumin_max,alt_max,bicarbonate_max,bun_max,creatinine_max,glucose_max,hematocrit_max,hemoglobin_max,inr_max,lactate_max,platelets_max,potassium_max,sodium_max,wbc_max,albumin_mean,alt_mean,...,sodium_min,wbc_min,dbp_max,hr_max,map_max,rr_max,sbp_max,spo2_max,temp_max,dbp_mean,hr_mean,map_mean,rr_mean,sbp_mean,spo2_mean,temp_mean,dbp_min,hr_min,map_min,rr_min,sbp_min,spo2_min,temp_min,icd9_chapter,dataset
0,206504,142345,70.000,0,African,1.633,MICU,Emergency,0,2.700,NaN,31.000,11.000,3.500,84.000,36.900,NaN,3.500,1.300,106.000,4.400,139.000,4.600,2.700,NaN,...,139.000,4.600,60.000,104.000,80.667,91.000,132.000,100.000,NaN,51.286,86.062,72.810,28.875,115.857,98.533,NaN,45.000,70.000,63.667,15.000,91.000,97.000,NaN,Injury & poisoning,mimic
1,232110,105331,36.000,0,Unknown,13.851,MICU,Transfer,1,2.600,254.000,23.000,3.000,0.700,79.000,34.000,NaN,9.500,NaN,299.000,5.900,136.000,10.600,2.600,254.000,...,136.000,10.600,71.000,108.000,86.000,23.000,119.000,100.000,NaN,48.542,86.958,67.611,18.833,105.750,98.913,NaN,38.000,72.000,57.667,14.000,97.000,97.000,NaN,Digestive,mimic
2,264446,165520,87.000,0,Unknown,2.650,MICU,Transfer,1,NaN,29.000,29.000,32.000,1.700,165.000,29.200,NaN,1.500,NaN,220.000,4.200,138.000,16.200,NaN,29.000,...,136.000,13.800,58.000,113.000,91.000,33.000,147.000,96.000,38.500,39.413,98.725,60.468,23.900,110.804,93.385,37.463,28.000,84.000,0.000,18.000,87.000,83.000,36.800,Infectious & parasitic,mimic


In [ ]:
# Missingness report
miss = mimic_features.isnull().mean().sort_values(ascending=False)
print('Missing rate per feature (top 20):')
print(miss[miss > 0].head(20).to_string())

Missing rate per feature (top 20):
temp_min          0.934
temp_mean         0.934
temp_max          0.934
hemoglobin_max    0.875
hemoglobin_mean   0.875
hemoglobin_min    0.875
albumin_mean      0.728
albumin_min       0.728
albumin_max       0.728
alt_mean          0.676
alt_max           0.676
alt_min           0.676
lactate_mean      0.544
lactate_min       0.544
lactate_max       0.544
inr_min           0.199
inr_mean          0.199
inr_max           0.199
hematocrit_min    0.037
wbc_mean          0.037


In [ ]:
out_path = os.path.join(MIMIC_OUT_DIR, 'mimic_features.csv')
mimic_features.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Shape: {mimic_features.shape[0]:,} rows × {mimic_features.shape[1]} cols')

Saved: /content/drive/MyDrive/AI in Medicine/data/output_data/mimic_val/mimic_features.csv
Shape: 136 rows × 74 cols


In [ ]:
out_path_2 = os.path.join(EICU_OUT_DIR, 'eicu_features.csv')
eicu_features.to_csv(out_path_2, index=False)
print(f'Saved: {out_path_2}')
print(f'Shape: {eicu_features.shape[0]:,} rows × {eicu_features.shape[1]} cols')

Saved: /content/drive/MyDrive/AI in Medicine/data/output_data/eicu_train/eicu_features.csv
Shape: 2,520 rows × 73 cols
